In [2]:
%pip install cassandra-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 3.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cassandra-driver][cassandra-driver]
Note: you may need to restart the kernel to use updated packages.


In [5]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("TP1-Decouverte-Spark")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

sc = spark.sparkContext

print("Version Spark :", spark.version)
print("Master :", sc.master)
print("Parallélisme :", sc.defaultParallelism)

Version Spark : 4.2.0
Master : local[4]
Parallélisme : 4


In [6]:
from cassandra.cluster import Cluster

cluster = Cluster(["cass1"])
session = cluster.connect()
print("Connexion Cassandra réussie")

Connexion Cassandra réussie


In [12]:
rows = session.execute("""
    SELECT keyspace_name
    FROM system_schema.keyspaces
""")

for row in rows:
    print(row.keyspace_name)

print("-----")

rows = session.execute("""
    SELECT table_name
    FROM system_schema.tables
    WHERE keyspace_name = 'meteo'
""")

for row in rows:
    print(row.table_name)

system_auth
system_schema
meteo
system_distributed
system
system_traces
-----
releves_temperature
villes


In [16]:
KEYSPACE = "meteo"
TABLES = {
    "temp": "releves_temperature",
    "villes": "villes"
}

session.set_keyspace(KEYSPACE)

rows = session.execute("""
    SELECT column_name, type
    FROM system_schema.columns
    WHERE keyspace_name = 'meteo'
    AND table_name = 'villes'
""")

for row in rows:
    print(row.column_name, "->", row.type)

print("----")

rows = session.execute("""
    SELECT column_name, type
    FROM system_schema.columns
    WHERE keyspace_name = 'meteo'
    AND table_name = 'releves_temperature'
""")

for row in rows:
    print(row.column_name, "->", row.type)

latitude -> double
longitude -> double
nom -> text
ville_id -> text
----
observed_at -> timestamp
temperature_2m -> double
ville_id -> text


In [18]:
rows = session.execute("""
    SELECT *
    FROM releves_temperature
    LIMIT 20
""")

for row in rows:
    print(row)

Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 12, 0), temperature_2m=22.2)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 13, 0), temperature_2m=23.5)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 14, 0), temperature_2m=24.2)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 15, 0), temperature_2m=24.2)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 16, 0), temperature_2m=24.1)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 17, 0), temperature_2m=23.4)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 18, 0), temperature_2m=22.1)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 19, 0), temperature_2m=21.1)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 20, 0), temperature_2m=20.3)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10, 6, 21, 0), temperature_2m=19.4)
Row(ville_id='paris', observed_at=datetime.datetime(2026, 10

In [20]:
rows = session.execute("""
    SELECT *
    FROM releves_temperature
""")

columns = rows.column_names
data = [tuple(row) for row in rows]

print("Colonnes :", columns)
print("Nombre de lignes récupérées :", len(data))

Colonnes : ['ville_id', 'observed_at', 'temperature_2m']
Nombre de lignes récupérées : 21


In [23]:
df = spark.createDataFrame(data, columns)

df.show(10, truncate=False)
df.printSchema()

print("Nombre de lignes dans le DataFrame :", df.count())

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-06 21:00:00|19.4          |
+--------+-------------------+--------------+
only showing top 10 rows
root
 |-- ville_id: string (nullable = true)
 |-- observed_at: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)

Nombre de lignes dans le DataFrame : 21


In [25]:
from pyspark.sql import functions as F

df.select(
    "ville_id",
    "observed_at",
    "temperature_2m"
).show(10, truncate=False)

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-06 21:00:00|19.4          |
+--------+-------------------+--------------+
only showing top 10 rows


In [28]:
# Stations avec moins de 5 vélos disponibles
df.filter(F.col("temperature_2m") < 5).show(10, truncate=False)
# Stations avec plus de 20 vélos disponibles
df.filter(F.col("temperature_2m") > 20).show(10, truncate=False)

+--------+-----------+--------------+
|ville_id|observed_at|temperature_2m|
+--------+-----------+--------------+
+--------+-----------+--------------+

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-20 12:00:00|99.9          |
+--------+-------------------+--------------+



## Données utilisées

Je travaille sur la table `releves_temperature` du keyspace `meteo`, qui contient des relevés horaires de température pour Paris, récupérés depuis l'API Open-Meteo.

## Colonnes importantes

| Colonne | Type | Rôle |
|---|---|---|
| `ville_id` | text | la ville concernée (clé de partition dans Cassandra) |
| `observed_at` | timestamp | l'heure du relevé (clé de clustering) |
| `temperature_2m` | double | la température de l'air à 2 m, en °C : c'est la valeur analysée |

## Information métier à analyser

Je souhaite analyser l'évolution de la température à Paris sur la période couverte par les relevés : repérer les heures les plus chaudes et les plus froides, calculer la température moyenne, et classer les relevés par niveau de température.

## Partie 3 - Transformations Spark

Sur le DataFrame df issu de la table releves_temperature, j'ai réalisé trois types de transformations :

- Sélection : je n'ai gardé que les colonnes utiles à l'analyse (ville_id, observed_at, temperature_2m).
- Filtres métier : un premier filtre garde les relevés dont la température dépasse un seuil (> 15 °C), et un second ceux d'une ville donnée (paris).
- Colonne calculée : ecart_moyenne, c'est-à-dire la température d'un relevé moins la température moyenne de la période. Elle indique si une heure est plus chaude (écart positif) ou plus froide (écart négatif) que la moyenne.

In [30]:
# Sélection : uniquement les colonnes utiles
df_select = df.select("ville_id", "observed_at", "temperature_2m")
df_select.show(10, truncate=False)

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-06 21:00:00|19.4          |
+--------+-------------------+--------------+
only showing top 10 rows


In [31]:
# Filtre 1 : relevés avec une température supérieure à un seuil
df_select.filter(F.col("temperature_2m") > 15).show(10, truncate=False)

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-06 21:00:00|19.4          |
+--------+-------------------+--------------+
only showing top 10 rows


In [32]:
# Filtre 2 : relevés d'une ville donnée
df_select.filter(F.col("ville_id") == "paris").show(10, truncate=False)

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
|paris   |2026-10-06 21:00:00|19.4          |
+--------+-------------------+--------------+
only showing top 10 rows


In [38]:
# Colonne calculée : écart de chaque relevé par rapport à la température moyenne
moyenne = df_select.agg(F.avg("temperature_2m")).first()[0]

df_ecart = df_select.withColumn(
    "ecart_moyenne",
    F.round(F.col("temperature_2m") - F.lit(moyenne), 2)
)

df_ecart.show(10, truncate=False)

+--------+-------------------+--------------+-------------+
|ville_id|observed_at        |temperature_2m|ecart_moyenne|
+--------+-------------------+--------------+-------------+
|paris   |2026-10-06 12:00:00|22.2          |-0.83        |
|paris   |2026-10-06 13:00:00|23.5          |0.47         |
|paris   |2026-10-06 14:00:00|24.2          |1.17         |
|paris   |2026-10-06 15:00:00|24.2          |1.17         |
|paris   |2026-10-06 16:00:00|24.1          |1.07         |
|paris   |2026-10-06 17:00:00|23.4          |0.37         |
|paris   |2026-10-06 18:00:00|22.1          |-0.93        |
|paris   |2026-10-06 19:00:00|21.1          |-1.93        |
|paris   |2026-10-06 20:00:00|20.3          |-2.73        |
|paris   |2026-10-06 21:00:00|19.4          |-3.63        |
+--------+-------------------+--------------+-------------+
only showing top 10 rows


## Partie 4 — Actions et agrégations

Question métier : « Combien de relevés appartiennent à chaque niveau de température (Froid, Doux, Chaud), et quelles sont la moyenne, le minimum et le maximum de chaque niveau ? »

- Seuils : 12 et 16 °C sont des exemples. Adapte-les à tes données (regarde ton minimum et ton maximum), sinon tous les relevés tomberont dans la même catégorie.
- Pourquoi pas un groupBy("ville_id") : ta table ne contient qu'une ville, donc le résultat serait une seule ligne. Regrouper par niveau est plus parlant.
- Actions : show() et count() déclenchent le calcul, alors que withColumn, groupBy et agg ne sont que des transformations. C'est utile pour la partie sur la Lazy Evaluation.

In [36]:
# Colonne de catégorie : niveau de température
df_categorie = df_select.withColumn(
    "niveau",
    F.when(F.col("temperature_2m") < 12, "Froid")
     .when(F.col("temperature_2m") <= 16, "Doux")
     .otherwise("Chaud")
)

# Action d'affichage
df_categorie.show(10, truncate=False)

# Action de comptage
print("Nombre de relevés :", df_categorie.count())

+--------+-------------------+--------------+------+
|ville_id|observed_at        |temperature_2m|niveau|
+--------+-------------------+--------------+------+
|paris   |2026-10-06 12:00:00|22.2          |Chaud |
|paris   |2026-10-06 13:00:00|23.5          |Chaud |
|paris   |2026-10-06 14:00:00|24.2          |Chaud |
|paris   |2026-10-06 15:00:00|24.2          |Chaud |
|paris   |2026-10-06 16:00:00|24.1          |Chaud |
|paris   |2026-10-06 17:00:00|23.4          |Chaud |
|paris   |2026-10-06 18:00:00|22.1          |Chaud |
|paris   |2026-10-06 19:00:00|21.1          |Chaud |
|paris   |2026-10-06 20:00:00|20.3          |Chaud |
|paris   |2026-10-06 21:00:00|19.4          |Chaud |
+--------+-------------------+--------------+------+
only showing top 10 rows
Nombre de relevés : 21


In [43]:
# Agrégation avec groupBy : nombre de relevés et statistiques par niveau
df_resultat = (
    df_categorie
    .groupBy("niveau")
    .agg(
        F.count("*").alias("nb_releves"),
        F.round(F.avg("temperature_2m"), 2).alias("temp_moyenne"),
        F.min("temperature_2m").alias("temp_min"),
        F.max("temperature_2m").alias("temp_max")
    )
    .orderBy(F.desc("nb_releves"))
)

spark.sparkContext.setJobDescription("Partie 4 : agrégation par niveau")
df_resultat.show()

+------+----------+------------+--------+--------+
|niveau|nb_releves|temp_moyenne|temp_min|temp_max|
+------+----------+------------+--------+--------+
| Chaud|        15|       26.27|    16.5|    99.9|
|  Doux|         6|       14.92|    14.3|    15.9|
+------+----------+------------+--------+--------+



## Partie 5

Pourquoi Spark utilise-t-il la Lazy Evaluation ?

Spark n'exécute pas chaque transformation tout de suite : il les enregistre dans un plan (le DAG) et attend une action (show, count, collect...) pour lancer le calcul. Cela présente plusieurs avantages :

Optimisation : comme Spark voit toute la chaîne avant d'exécuter, il peut la réorganiser et fusionner les étapes (par exemple appliquer un filtre le plus tôt possible).
Moins de calculs inutiles : seules les données nécessaires au résultat demandé sont calculées, et il n'y a pas de résultats intermédiaires stockés pour rien.
Moins de mémoire et de transferts : les étapes enchaînées sont exécutées ensemble, partition par partition.
Tolérance aux pannes : le plan décrit comment reconstruire les données, donc Spark peut recalculer une partition perdue.

In [44]:
# 1. Nombre de partitions actuel
print("Partitions initiales :", df.rdd.getNumPartitions())

# 2. Modifier le nombre de partitions
df2 = df.repartition(2)
print("Après repartition(2) :", df2.rdd.getNumPartitions())

df8 = df.repartition(8)
print("Après repartition(8) :", df8.rdd.getNumPartitions())

# 3. Transformations successives (aucun calcul à ce stade)
result = (
    df8
    .filter(F.col("temperature_2m") > 12)
    .select("ville_id", "observed_at", "temperature_2m")
    .orderBy(F.desc("temperature_2m"))
)

print("Le plan logique est construit, mais aucun calcul n'a été exécuté.")

# 4. Action : déclenche l'exécution
spark.sparkContext.setJobDescription("Partie 5 : result.show(10)")
result.show(10, truncate=False)

Partitions initiales : 4
Après repartition(2) : 2
Après repartition(8) : 8
Le plan logique est construit, mais aucun calcul n'a été exécuté.
+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-20 12:00:00|99.9          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
+--------+-------------------+--------------+
only showing top 10 rows


## Interface Spark UI

In [46]:
#df_resultat est l'agrégation par niveau de température de la partie 4. explain(True) affiche le plan, show() déclenche le calcul et crée un Job.


spark.sparkContext.setJobDescription("Partie 4 : agrégation par niveau")
df_resultat.show()

spark.sparkContext.setJobDescription("Partie 5 : tri après repartition(8)")
result.show(10, truncate=False)

df_resultat.explain(True)
df_resultat.show()

+------+----------+------------+--------+--------+
|niveau|nb_releves|temp_moyenne|temp_min|temp_max|
+------+----------+------------+--------+--------+
| Chaud|        15|       26.27|    16.5|    99.9|
|  Doux|         6|       14.92|    14.3|    15.9|
+------+----------+------------+--------+--------+

+--------+-------------------+--------------+
|ville_id|observed_at        |temperature_2m|
+--------+-------------------+--------------+
|paris   |2026-10-20 12:00:00|99.9          |
|paris   |2026-10-06 14:00:00|24.2          |
|paris   |2026-10-06 15:00:00|24.2          |
|paris   |2026-10-06 16:00:00|24.1          |
|paris   |2026-10-06 13:00:00|23.5          |
|paris   |2026-10-06 17:00:00|23.4          |
|paris   |2026-10-06 12:00:00|22.2          |
|paris   |2026-10-06 18:00:00|22.1          |
|paris   |2026-10-06 19:00:00|21.1          |
|paris   |2026-10-06 20:00:00|20.3          |
+--------+-------------------+--------------+
only showing top 10 rows
== Parsed Logical Plan =

In [47]:
#L'interface de spark UI refusant de répondre et étant absolument ni claire ni lisible

sc = spark.sparkContext
tracker = sc.statusTracker()

sc.setJobGroup("p4", "Partie 4 : agrégation par niveau")
df_resultat.show()

for job_id in tracker.getJobIdsForGroup("p4"):
    job = tracker.getJobInfo(job_id)
    print("Job", job_id, "-", job.status, "- stages :", job.stageIds)
    for stage_id in job.stageIds:
        stage = tracker.getStageInfo(stage_id)
        if stage:
            print("   Stage", stage_id, ": tasks =", stage.numTasks)

+------+----------+------------+--------+--------+
|niveau|nb_releves|temp_moyenne|temp_min|temp_max|
+------+----------+------------+--------+--------+
| Chaud|        15|       26.27|    16.5|    99.9|
|  Doux|         6|       14.92|    14.3|    15.9|
+------+----------+------------+--------+--------+

Job 91 - SUCCEEDED - stages : [I@7d20b914
   Stage 112 : tasks = 4
   Stage 113 : tasks = 1
Job 90 - SUCCEEDED - stages : [I@16ac7e83
   Stage 111 : tasks = 4


## Partie 7 — Mon analyse métier

### Pourquoi ces données
J'ai choisi la table releves_temperature du keyspace meteo, qui contient des relevés horaires de température à Paris (ville_id, observed_at, temperature_2m). Elle a une seule colonne de mesure et une dimension temporelle, ce qui permet de savoir à quel moment la température est élevée ou basse.

### Pourquoi ces transformations

J'ai fait une sélection pour ne garder que les colonnes utiles, afin que Spark manipule moins de données.
J'ai fait un filtre sur la température pour écarter les relevés qui n'intéressent pas l'analyse.
J'ai créé une colonne calculée (niveau) qui transforme une valeur brute en information métier : « 22.2 » devient « Chaud ».

Pourquoi mon agrégation est pertinente
Je regroupe par niveau avec groupBy, puis je calcule le nombre de relevés, la moyenne, le minimum et le maximum de chaque niveau. Cela répond à ma question métier : combien d'heures sont Froid, Doux ou Chaud, et à quelles températures. Regrouper par ville n'aurait donné qu'une seule ligne, puisque ma table ne contient que Paris.

Ce qui déclenche réellement l'exécution
select, filter, withColumn, groupBy et agg sont des transformations : elles construisent seulement un plan, sans calcul. Le calcul démarre quand j'appelle une action comme show() ou count(). La lecture Cassandra avec session.execute(...) s'exécute à part, tout de suite, côté Python. C'est à partir de createDataFrame que Spark devient paresseux.

Ce que j'observe dans la Spark UI

Le job de mon agrégation compte 2 stages : le stage 112 a 4 tasks (une par partition) et le stage 113 en a 1.
Ces 2 stages viennent du groupBy, qui oblige à regrouper les relevés de même niveau au même endroit. C'est le shuffle, visible dans le plan avec explain() sous le nom Exchange. Il apparaît dans la Spark UI comme Shuffle Write en fin de stage 1 et Shuffle Read au début du stage 2.
Le job sans agrégation n'a qu'1 stage de 4 tasks, car il n'a pas besoin de redistribuer les données.